### Importing packages

In [55]:
# importing imp packages
import os
import zipfile
import random
import yaml
from google.cloud import storage
import uuid
from datetime import date


### From training.ipynb

In [7]:
def download_data_from_gcs(bucket_name: str, gcs_path: str, local_dir: str) -> None:
    """
    Downloads a directory and its contents from a Google Cloud Storage bucket.

    Args:
        bucket_name (str): The name of the GCS bucket.
        gcs_path (str): The path to the directory in GCS (e.g., "datasets/my_data/").
        local_dir (str): The local directory to save the downloaded files.
    """
    print(
        f"Attempting to download data from gs://{bucket_name}/{gcs_path} to {local_dir}"
    )

    try:
        storage_client = storage.Client()
        bucket = storage_client.bucket(bucket_name)

        # Ensure local directory exists
        os.makedirs(local_dir, exist_ok=True)

        blobs = bucket.list_blobs(
            prefix=gcs_path
        )  # List all blobs with the given prefix
        downloaded_count = 0
        for blob in blobs:
            # Skip blobs that represent directories themselves (ending with '/')
            # And filter for files that end with '.zip'
            if not blob.name.endswith("/") and blob.name.endswith(".zip"):
                # Construct local file path, preserving relative directory structure if any
                local_file_name = os.path.basename(blob.name)
                local_file_path = os.path.join(local_dir, local_file_name)

                blob.download_to_filename(local_file_path)
                downloaded_count += 1
            else:
                print(f"Skipping non-zipped file or directory: {blob.name}")

        if downloaded_count == 0:
            print(
                f"No zipped files found or downloaded from gs://{bucket_name}/{gcs_path}. "
                f"Please check bucket name and GCS path, and ensure there are .zip files present."
            )
        else:
            print(f"Successfully downloaded {downloaded_count} zipped files from GCS.")

    except Exception as e:
        print(f"An error occurred: {e}")
        print(f"Error downloading data from GCS: {e}")
        print("Please ensure your Google Cloud credentials are set up correctly.")
        print(
            "You can use `gcloud auth application-default login` for local development or set the "
            "`GOOGLE_APPLICATION_CREDENTIALS` environment variable for service accounts."
        )
        exit(1)  # Exit if data download fails
# unzipping files

def unzipDataset(folderPath: str) -> None:
    """
    Unzips all .zip files in the specified folder.
    Args:
        folderPath (str): The path to the folder containing .zip files.
    """
    files = os.listdir(folderPath)
    for file in files:
        fullPath = os.path.join(folderPath, file)
        # Check if the item is a file and ends with .zip
        if os.path.isfile(fullPath) and fullPath.endswith(".zip"):
            filename = file.split(".")[0]
            try:
                with zipfile.ZipFile(fullPath, "r") as zip_ref:
                    # Extract to a subdirectory with the same name as the zip file
                    extract_dir = os.path.join(folderPath, filename)
                    print(f"Unzipping {file} into {extract_dir}")
                    zip_ref.extractall(extract_dir)

                # Remove the zip file after successful extraction
                os.remove(fullPath)
            except Exception as e:
                print(f"Error unzipping or removing file {fullPath}: {e}")

### Configurations

In [22]:
GCS_BUCKET_NAME = "open-cityvision"
GCS_DATA_PATH = "Dataset/" # Path to the zipped files in GCS (e.g., 'raw_data/')
GCS_DESTINATION_PREFIX = "Split_Dataset/" # Path where the new combined dataset will be uploaded

LOCAL_DATA_DIR = "yolo_dataset/" # Local directory to download and process files

# Define your class names here in the correct order. This is crucial for training.
CLASS_NAMES = ["class1", "class2", "class3"] # Example: ["cat", "dog", "car"]

# Define the split ratios for your dataset.
TRAIN_SPLIT_RATIO = 0.8
VAL_SPLIT_RATIO = 0.1
TEST_SPLIT_RATIO = 0.1


### New function

In [62]:
def get_class_names_from_yaml(local_dir: str) -> list:
    """
    Finds and extracts class names from the first data.yaml file found.
    """
    for root, _, files in os.walk(local_dir):
        if "data.yaml" in files:
            yaml_path = os.path.join(root, "data.yaml")
            print(f"Found data.yaml at {yaml_path}. Reading class names...")
            with open(yaml_path, 'r') as f:
                data = yaml.safe_load(f)
                if 'names' in data:
                    print("Successfully extracted class names.")
                    return data['names']
    
    print("Warning: No 'data.yaml' file found to extract class names. Returning an empty list.")
    return []

def combine_and_split_dataset(source_dir, train_ratio, val_ratio, test_ratio):
    """
    Combines images and labels from multiple subdirectories and splits them.
    """
    print("Combining and splitting dataset...")
    
    
    # Collect all image and label files
    all_image_paths = []
    all_label_paths = {} # New dictionary to map image filenames to label paths
    print("Collecting image and label files...")
    for root, dirs, files in os.walk(source_dir):
        for file in files:
            # We are looking for image files (e.g., .jpg, .png)
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                # Check for the images folder in the path
                if "images" in root.split(os.path.sep):
                    image_path = os.path.join(root, file)
                    # check if the image path exists
                    if not os.path.exists(image_path):
                        print(f"Warning: Image file {image_path} does not exist. Skipping.")
                        continue
                    # remove "images" word from the path and replace it with "labels"
                    label_root = root.replace("images", "labels")
                    label_path = os.path.join(label_root, file.split('.')[0] + '.txt')
                    # if the label file exists, add it to the dictionary
                    if os.path.exists(label_path):
                        # mapping image to label 
                        all_label_paths[image_path] = [label_root, label_path]
                        # only append the image path if the label exists
                        all_image_paths.append(image_path)

    if not all_image_paths:
        print("No image files found. Please check your source directory and file extensions.")
        return

    # Shuffle the list of images to ensure random distribution
    random.shuffle(all_image_paths)
    # Define new directory structure
    
    Today = date.today()
    output_dir = os.path.join(source_dir, f"{Today:%Y-%m-%d}_len_{len(all_image_paths)}")
    # create output directory if doesnt exist
    os.makedirs(output_dir, exist_ok=True)
    train_images_dir = os.path.join(output_dir, "train", "images")
    train_labels_dir = os.path.join(output_dir, "train", "labels")
    val_images_dir = os.path.join(output_dir, "val", "images")
    val_labels_dir = os.path.join(output_dir, "val", "labels")
    test_images_dir = os.path.join(output_dir, "test", "images")
    test_labels_dir = os.path.join(output_dir, "test", "labels")

    # Create directories
    for d in [train_images_dir, train_labels_dir, val_images_dir, val_labels_dir, test_images_dir, test_labels_dir]:
        os.makedirs(d, exist_ok=True)
    # Calculate split indices
    num_images = len(all_image_paths)
    num_train = int(num_images * train_ratio)
    num_val = int(num_images * val_ratio)
    
    # Split the dataset
    train_images = all_image_paths[:num_train]
    val_images = all_image_paths[num_train : num_train + num_val]
    test_images = all_image_paths[num_train + num_val:]

    # Helper function to move files with unique filenames
    def move_files(image_paths, image_dest, label_dest):
        for img_path in image_paths:
            label = all_label_paths.get(img_path, None)
            label_root, label_path = label if label else (None, None)
            # get the extension of the image file
            _, ext = os.path.splitext(os.path.basename(img_path))

            # Check if corresponding label file exists before moving
            if label_path and os.path.exists(label_path):
                # Generate a unique name for the file
                unique_filename = f"{uuid.uuid4()}{ext}"
                
                # Rename and move the image file
                os.rename(img_path, os.path.join(image_dest, unique_filename))
                # Rename and move the label file to match the new image name
                os.rename(label_path, os.path.join(label_dest, os.path.splitext(unique_filename)[0] + ".txt"))
            else:
                print(f"Warning: Label file for {img_path} not found. Skipping.")

    print(f"Moving {len(train_images)} files to train set.")
    move_files(train_images, train_images_dir, train_labels_dir)
    print(f"Moving {len(val_images)} files to validation set.")
    move_files(val_images, val_images_dir, val_labels_dir)
    print(f"Moving {len(test_images)} files to test set.")
    move_files(test_images, test_images_dir, test_labels_dir)
    
    print("Dataset combination and splitting complete.")
    return output_dir



def create_data_yaml(output_dir, class_names):
    """
    Creates the data.yaml file required for YOLOv8 training.
    """
    print("Creating data.yaml file...")
    path = output_dir.split(os.path.sep)[-1]  # Get the last part of the output directory path
    data = {
        'path': f'{path}',
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images', # Optional
        'nc': len(class_names),
        'names': class_names
    }
    
    yaml_file_path = os.path.join(output_dir, "data.yaml")
    with open(yaml_file_path, 'w') as f:
        yaml.dump(data, f, sort_keys=False)
        
    print(f"data.yaml created at {yaml_file_path}")

def upload_combined_dataset(local_dir, bucket_name, destination_prefix):
    """
    Uploads the entire processed dataset folder to GCS.
    """
    print(f"Uploading combined dataset to GCS bucket: {bucket_name}, prefix: {destination_prefix}...")
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)

    for root, _, files in os.walk(local_dir):
        for file in files:
            local_file_path = os.path.join(root, file)
            # Create a GCS destination path that maintains the folder structure
            relative_path = os.path.relpath(local_file_path, local_dir)
            gcs_path = os.path.join(destination_prefix, relative_path).replace("\\", "/") # Use forward slashes
            
            blob = bucket.blob(gcs_path)
            print(f"Uploading {local_file_path} to {gcs_path}")
            blob.upload_from_filename(local_file_path)
            
    print("Upload complete.")




In [ ]:
if __name__ == "__main__":
    # clean the folders if they exist
    if os.path.exists(LOCAL_DATA_DIR):
        print(f"Cleaning up existing local directory: {LOCAL_DATA_DIR}")
        for root, dirs, files in os.walk(LOCAL_DATA_DIR, topdown=False):
            for name in files:
                os.remove(os.path.join(root, name))
            for name in dirs:
                os.rmdir(os.path.join(root, name))
        os.rmdir(LOCAL_DATA_DIR)
    
    # Step 1: Download from GCS using the new function
    download_data_from_gcs(GCS_BUCKET_NAME, GCS_DATA_PATH, LOCAL_DATA_DIR)

    # Step 2: Unzip the downloaded files using the new function
    unzipDataset(LOCAL_DATA_DIR)

    # Step 3: Get class names from a data.yaml file
    class_names = get_class_names_from_yaml(LOCAL_DATA_DIR)

    # Step 4: Combine and split the dataset
    combined_dataset_output = combine_and_split_dataset(LOCAL_DATA_DIR, TRAIN_SPLIT_RATIO, VAL_SPLIT_RATIO, TEST_SPLIT_RATIO)
    
    # Step 5: Create the data.yaml file with the extracted class names
    create_data_yaml(combined_dataset_output, class_names)
    
    # Step 6: Upload the new dataset to GCS
    upload_combined_dataset(combined_dataset_output, GCS_BUCKET_NAME, GCS_DESTINATION_PREFIX)

    print("\nDataset preparation and upload process finished.")

Cleaning up existing local directory: yolo_dataset/
Attempting to download data from gs://open-cityvision/Dataset/ to yolo_dataset/
Skipping non-zipped file or directory: Dataset/
Successfully downloaded 2 zipped files from GCS.
Unzipping June_10_2025_2871.zip into yolo_dataset/June_10_2025_2871
Unzipping June_18_2025_1108.zip into yolo_dataset/June_18_2025_1108
Found data.yaml at yolo_dataset/June_10_2025_2871\June_10_2025_2871\data.yaml. Reading class names...
Successfully extracted class names.
Combining and splitting dataset...
Moving 3066 files to train set.
Moving 383 files to validation set.
Moving 384 files to test set.
Dataset combination and splitting complete.
Creating data.yaml file...
data.yaml created at yolo_dataset/2025-08-04_len_3833\data.yaml

Dataset preparation and upload process finished.
